# เทียบ เก่า (v4 เดิม) vs ใหม่ (v4 + ระบบผสมอารมณ์)
ต้องรัน 2 server พร้อมกัน (2 Terminal):
- ใหม่: `./TTS-Lab/run_tts_native.sh` (port 8001)
- เก่า: `./TTS-Lab/run_tts_native_old.sh` (port 8002, `SPEECH_MIX=0` = v4 เดิม tag + PSOLA)

หมายเหตุ: ทั้งสองใช้ `tts.py` ตัวที่แก้ clip แล้ว (เก่า = วิธีอารมณ์เดิม ไม่ใช่โค้ดเดิมเป๊ะ)

In [ ]:
import requests
from IPython.display import Audio, display, HTML
NEW, OLD = "http://localhost:8001", "http://localhost:8002"
for n,u in (("ใหม่",NEW),("เก่า",OLD)):
    print(n, requests.get(u+"/v1/health").json())

def speak(base, text, voice, emotion, intensity=0.6, speed=1.0):
    r = requests.post(base+"/v1/speak", json=dict(text=text, voice=voice, emotion=emotion,
                      intensity=intensity, speed=speed), timeout=600)
    r.raise_for_status()
    return r.content, r.headers.get("x-engine")

def compare(text, voice, emotion, intensity=0.6):
    o,eo = speak(OLD, text, voice, emotion, intensity)
    n,en = speak(NEW, text, voice, emotion, intensity)
    display(HTML(f"<b>{emotion}</b> · {voice} · {text}"
                 f"<table><tr><td>เก่า <small>({eo})</small></td><td>ใหม่ <small>({en})</small></td></tr></table>"))
    display(Audio(o)); display(Audio(n))

## 1) ประโยคเดียว ทุกอารมณ์ (เก่า → ใหม่ ทีละคู่)

In [ ]:
TEXT, VOICE = "วันนี้ฉันดีใจมาก", "female"
for e in ["happy","sad","angry","fear","disgust","surprise"]:
    compare(TEXT, VOICE, e)

## 2) ประโยคที่ความหมายตรงอารมณ์ × หญิง/ชาย

In [ ]:
TESTS = [("วันนี้ฉันดีใจมากที่สอบผ่าน","happy"), ("ฉันเสียใจมากที่เขาจากไป","sad"),
         ("ทำไมทำแบบนี้ ฉันโมโหมากแล้วนะ","angry")]
for text, emo in TESTS:
    for v in ["female","male"]:
        compare(text, v, emo)

## 3) ลองเอง

In [ ]:
compare("ขอบคุณมากครับ", "male", "happy", 0.6)